# atividade pratica 01 sklearn pytorch basics 

**Status:** Em andamento · **Área:** classificacao-digitos

Fluxo mais completo da família Digits: divisão estratificada, scaler ajustado no treino, regressão logística, MLP, métricas e análise de erros.

Este arquivo está identificado como **Em andamento**. A organização e a revisão estática não equivalem à validação completa da análise. Consulte as pendências antes de usar seus resultados.

## Como retomar

1. Leia o README desta pasta e suas pendências.
2. Prepare os dados conforme data/README.md quando existir.
3. Reinicie o ambiente e execute em ordem; registre a verificação no README.

[Catálogo](https://github.com/Inocentrax/portfolio-dados/blob/main/CATALOGO.md) · [Plano de continuidade](https://github.com/Inocentrax/portfolio-dados/blob/main/PROXIMOS_PASSOS.md)

## Origem e créditos

Importado de um notebook pessoal do Drive em 15/11/2026. Créditos de cursos, livros e coautores no material original foram preservados. Outputs e metadados pessoais do Colab foram retirados.

---

## Desenvolvimento


# Atividade prática — do ML clássico a uma MLP

**Semanas 01 e 02 · Dataset Digits · Scikit-learn + PyTorch**

Nesta atividade você irá carregar e inspecionar dados, separar treino e teste, normalizar sem vazamento, treinar uma regressão logística e uma rede MLP, calcular métricas e analisar erros.

**Regras para a correção automática**

- Não altere nomes, parâmetros ou retornos das funções.
- Não use o conjunto de teste para ajustar o normalizador ou treinar modelos.
- Mantenha `random_state=42` e as sementes indicadas.
- CPU é suficiente. Em *Ambiente de execução*, selecione **Executar tudo** antes de entregar.
- A pontuação total é 10,0. Consulte a tabela no fim do notebook.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("PyTorch:", torch.__version__)

## Exercício 1 — Carregamento e exploração

Carregue o dataset `Digits`. Retorne `X` como matriz `(n_amostras, 64)` e `y` como vetor de inteiros. Depois visualize algumas imagens: cada linha possui os 64 pixels de uma imagem 8×8.

In [ ]:
def load_digits_data():
    # TODO: carregue o dataset e retorne X e y.
    digits = load_digits()
    X = digits.data
    y = digits.target
    return X, y

In [ ]:
X, y = load_digits_data()
print("X:", X.shape, "y:", y.shape, "classes:", np.unique(y))

fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, image, label in zip(axes, X[:5], y[:5]):
    ax.imshow(image.reshape(8, 8), cmap="gray")
    ax.set_title(f"y={label}")
    ax.axis("off")
plt.tight_layout()

## Exercício 2 — Divisão estratificada

Separe 80% para treino e 20% para teste, com estratificação por classe e `random_state=42`. (Dica train_test_split)

In [ ]:
def split_data(X, y):
    # TODO: retorne X_train, X_test, y_train, y_test.
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
    return X_train, X_test, y_train, y_test

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## Exercício 3 — Normalização sem vazamento

Ajuste `StandardScaler` **somente no treino** (https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html). Retorne treino transformado, teste transformado e o scaler ajustado.

In [ ]:
def normalize_data(X_train, X_test):
    # TODO: ajuste no treino e transforme treino e teste.
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  return X_train_scaled, X_test_scaled, scaler

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)
X_train_scaled, X_test_scaled, scaler = normalize_data(X_train, X_test)
print("Treino/teste:", X_train_scaled.shape, X_test_scaled.shape)

## Exercício 4 — Regressão logística

Crie e ajuste `LogisticRegression` com `max_iter=1000` e `random_state=42`. Retorne o modelo treinado.

In [ ]:
def train_logistic(X_train, y_train):
    # TODO: instancie, ajuste e retorne o modelo.
    modelo = LogisticRegression(max_iter=1000, random_state=42)
    modelo.fit(X_train, y_train)
    return modelo

In [ ]:
logreg = train_logistic(X_train_scaled, y_train)
logreg_pred = logreg.predict(X_test_scaled)
print(f"Acurácia regressão logística: {accuracy_score(y_test, logreg_pred):.4f}")

## Exercício 5 — Arquitetura da MLP

Implemente `DigitMLP`: entrada 64, duas camadas ocultas de 64 e 32 neurônios com ReLU, e saída com 10 logits. Não aplique softmax: `CrossEntropyLoss` já o incorpora de forma numericamente estável.

In [ ]:
class DigitMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: defina as camadas em self.network.
        self.network = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x):
        # TODO: compute os logits.
        return self.network(x)

In [ ]:
modelo = DigitMLP()
entrada_teste = torch.randn(1, 64)
saida_teste = modelo(entrada_teste)

print(saida_teste.shape)

## Exercício 6 — DataLoaders e treinamento

Converta os dados para `float32` e rótulos para `long`. Treine em CPU com Adam e entropia cruzada. Retorne o modelo e um histórico `{'loss': [...]}` com uma perda média por época.

In [ ]:
def make_loaders(X_train, y_train, X_test, y_test, batch_size=64):
    # TODO: crie TensorDataset e DataLoader. Em treino, use shuffle=True.
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)

    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, test_loader

def train_pytorch_model(train_loader, epochs=20, lr=1e-3):
    # TODO: use DigitMLP, CrossEntropyLoss e Adam.
    # A loss de cada época deve ser a média ponderada pelo tamanho dos lotes.
    model = DigitMLP()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"loss": []}

    for epoch in range(epochs):
        model.train()

        total_loss = 0.0
        total_samples = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()

            batch_size = X_batch.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

        epoch_loss = total_loss / total_samples
        history["loss"].append(epoch_loss)

    return model, history


In [ ]:
train_loader, test_loader = make_loaders(
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    batch_size=64
)

print("Lotes de treino:", len(train_loader))
print("Lotes de teste:", len(test_loader))

In [ ]:
torch.manual_seed(SEED)
train_loader, test_loader = make_loaders(
    X_train_scaled, y_train, X_test_scaled, y_test, batch_size=64
)
mlp, history = train_pytorch_model(train_loader, epochs=20, lr=1e-3)
plt.plot(history["loss"])
plt.xlabel("Época"); plt.ylabel("Loss média"); plt.title("Treinamento da MLP"); plt.grid(alpha=.3)

## Exercício 7 — Predição e métricas

Implemente predição em modo de avaliação, sem gradientes. Depois retorne acurácia e matriz de confusão 10×10.

In [ ]:
def predict_pytorch(model, data_loader):
    # TODO: retorne um np.ndarray 1D com as classes previstas.
    model.eval()
    predictions = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            outputs = model(X_batch)
            predicted = torch.argmax(outputs, dim=1)
            predictions.extend(predicted.cpu().numpy())

    return np.array(predictions)

def compute_metrics(y_true, y_pred):
    # TODO: retorne {'accuracy': float, 'confusion_matrix': ndarray}.
    accuracy = accuracy_score(y_true, y_pred)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(10)
    )

    return {
        "accuracy": float(accuracy),
        "confusion_matrix": cm
    }

In [ ]:
mlp_pred = predict_pytorch(mlp, test_loader)
logreg_metrics = compute_metrics(y_test, logreg_pred)
mlp_metrics = compute_metrics(y_test, mlp_pred)
print("Logística:", logreg_metrics["accuracy"])
print("MLP:", mlp_metrics["accuracy"])

## Exercício 8 — Análise de erro

Retorne os índices errados, os pares `(verdadeiro, previsto)` e os três pares de confusão mais frequentes. Em um par de confusão, a ordem importa; exclua a diagonal.

In [ ]:
from collections import Counter

def analyze_errors(y_true, y_pred, top_k=3):
    # TODO: retorne um dicionário com error_indices, error_pairs e top_confusions.
    error_indices = np.where(y_true != y_pred)[0]

    error_pairs = [
        (int(y_true[i]), int(y_pred[i]))
        for i in error_indices
    ]

    counts = Counter(error_pairs)

    top_confusions = [
        (real, previsto, quantidade)
        for (real, previsto), quantidade
        in counts.most_common(top_k)
    ]

    return {
        "error_indices": error_indices,
        "error_pairs": error_pairs,
        "top_confusions": top_confusions
    }


In [ ]:
errors = analyze_errors(y_test, mlp_pred, top_k=3)
print("Total de erros:", len(errors["error_indices"]))
print("Confusões mais frequentes (real, previsto, quantidade):", errors["top_confusions"])

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, idx in zip(axes.ravel(), errors["error_indices"][:10]):
    ax.imshow(X_test[idx].reshape(8, 8), cmap="gray")
    ax.set_title(f"real={y_test[idx]} / pred={mlp_pred[idx]}")
    ax.axis("off")
plt.tight_layout()

## Continuidade

Priorizar como base principal. Trocar linguagem de entrega escolar por narrativa do experimento, preservar créditos, remover TODOs já resolvidos, registrar versões e sementes e escrever a comparação final baseada nas métricas de teste. Verificar execução limpa.

Registre decisões e resultados verificados no README desta pasta. Evite criar “Untitled” ou “Cópia de” para continuar este trabalho; use o mesmo arquivo e o histórico do Git.